# Best Time to Visit — Hourly Forecaster with Plotly Mini Dashboard (Business Hours)


This notebook:
- Loads **3 months** of hourly traffic data (CSV) or generates it if missing.
- Restricts to **Jamaica-style business hours** (default **09:00–17:00**, configurable).
- Engineers **weekday / week-of-month** and **cyclical** time features.
- Trains a **RandomForestRegressor** on historical data.
- Produces a **Pretty Dashboard** (text) and a **Plotly mini dashboard** (line chart + table).
- Returns **one final best time**; if ties are indistinguishable, shows **up to 4** best times.


In [ ]:

# 0) Setup & Config
import os, io, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

import plotly.graph_objs as go
from plotly.subplots import make_subplots

sns.set_theme(style="whitegrid")

DATA_PATH = Path("best_time_to_visit_sample_data.csv")  # sample CSV (3 months)
BUSINESS_START = "09:00"  # or "08:30"
BUSINESS_END   = "17:00"  # or "16:30" / "17:00"
TOP_K = 4                 # maximum number of options to display if ties remain
TIE_EPS = 0.15            # tolerance (in predicted traffic units) to consider ties indistinguishable

weekday_names = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

def to_ampm_from_hour(hour_int):
    ampm = "AM" if hour_int < 12 else "PM"
    h = hour_int % 12 or 12
    return f"{h}:00 {ampm}"


## 1) Load (or generate) 3‑month hourly data

In [2]:

if DATA_PATH.exists():
    df_hourly = pd.read_csv(DATA_PATH, parse_dates=["timestamp"]).sort_values("timestamp")
else:
    # Generate 3 months of hourly data (we will filter to business hours later)
    rng = np.random.default_rng(42)
    start_date = pd.Timestamp("2025-03-01 08:00")
    end_date   = pd.Timestamp("2025-05-31 20:00")
    timestamps = pd.date_range(start=start_date, end=end_date, freq="H")
    df_hourly = pd.DataFrame({"timestamp": timestamps})
    df_hourly["dow"] = df_hourly["timestamp"].dt.dayofweek
    df_hourly["hour"] = df_hourly["timestamp"].dt.hour
    df_hourly["is_weekend"] = df_hourly["dow"].isin([5,6]).astype(int)
    base = 40
    lunch_bump = np.where(df_hourly["hour"].between(12, 14), 1.3, 1.0)
    evening_bump = np.where(df_hourly["hour"].between(16, 18), 1.2, 1.0)
    weekend_bump = np.where(df_hourly["is_weekend"]==1, 1.4, 1.0)
    month_factor = (df_hourly["timestamp"].dt.month - 2) * 0.05 + 1.0
    noise = rng.normal(0, 8, size=len(df_hourly))
    traffic = base * lunch_bump * evening_bump * weekend_bump * month_factor + noise
    df_hourly["traffic"] = np.clip(traffic, 5, None).round(0)

# Ensure required columns if loaded from CSV
if "traffic" not in df_hourly.columns:
    raise ValueError("CSV must contain 'timestamp' and 'traffic' columns.")

df_hourly = df_hourly.sort_values("timestamp").reset_index(drop=True)
print("Date range:", df_hourly['timestamp'].min(), "→", df_hourly['timestamp'].max())
print("Rows:", len(df_hourly))
df_hourly.head(8)


Date range: 2025-03-01 08:00:00 → 2025-05-31 20:00:00
Rows: 1196


,timestamp,dow,hour,is_weekend,traffic
0,2025-03-01 08:00:00,5,8,1,61.0
1,2025-03-01 09:00:00,5,9,1,50.0
2,2025-03-01 10:00:00,5,10,1,65.0
3,2025-03-01 11:00:00,5,11,1,66.0
4,2025-03-01 12:00:00,5,12,1,61.0
5,2025-03-01 13:00:00,5,13,1,66.0
6,2025-03-01 14:00:00,5,14,1,77.0
7,2025-03-01 15:00:00,5,15,1,56.0


## 2) Business hours filter (hourly granularity)

In [3]:

# Filter to BUSINESS_START..BUSINESS_END inclusive by hour
bs_h, bs_m = map(int, BUSINESS_START.split(":"))
be_h, be_m = map(int, BUSINESS_END.split(":"))
df_biz = df_hourly[
    (df_hourly['timestamp'].dt.hour >= bs_h) & (df_hourly['timestamp'].dt.hour <= be_h)
].copy()

print("Business hours:", BUSINESS_START, "→", BUSINESS_END)
df_biz.head(6)


Business hours: 09:00 → 17:00


,timestamp,dow,hour,is_weekend,traffic
1,2025-03-01 09:00:00,5,9,1,50.0
2,2025-03-01 10:00:00,5,10,1,65.0
3,2025-03-01 11:00:00,5,11,1,66.0
4,2025-03-01 12:00:00,5,12,1,61.0
5,2025-03-01 13:00:00,5,13,1,66.0
6,2025-03-01 14:00:00,5,14,1,77.0


## 3) Feature engineering (weekday, week of month, cyclical encodings)

In [4]:

def week_of_month(dt):
    first = dt.replace(day=1)
    dom = dt.day
    adj = (first.weekday() + 1) % 7  # align Mon=0
    return int(np.ceil((dom + adj) / 7.0))

df_biz["dow"] = df_biz["timestamp"].dt.dayofweek
df_biz["hour"] = df_biz["timestamp"].dt.hour
df_biz["month"] = df_biz["timestamp"].dt.month
df_biz["week_of_month"] = df_biz["timestamp"].apply(week_of_month)
df_biz["is_weekend"] = df_biz["dow"].isin([5,6]).astype(int)

# Cyclical encodings (hour and dow)
df_biz["hour_sin"] = np.sin(2*np.pi*df_biz["hour"]/24)
df_biz["hour_cos"] = np.cos(2*np.pi*df_biz["hour"]/24)
df_biz["dow_sin"]  = np.sin(2*np.pi*df_biz["dow"]/7)
df_biz["dow_cos"]  = np.cos(2*np.pi*df_biz["dow"]/7)

features = ["dow","hour","month","week_of_month","is_weekend","hour_sin","hour_cos","dow_sin","dow_cos"]
target = "traffic"


## 4) Train / Test split (chronological) & model training

In [5]:

df_biz = df_biz.sort_values("timestamp").reset_index(drop=True)
cut = int(len(df_biz)*0.8)
train = df_biz.iloc[:cut].copy()
test  = df_biz.iloc[cut:].copy()

X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

model = RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

test_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, test_pred)
r2  = r2_score(y_test, test_pred)

df_biz["predicted"] = model.predict(df_biz[features])

print(f"Test MAE: {mae:.2f}")
print(f"Test R² : {r2:.3f}")


Test MAE: 6.66
Test R² : 0.664


## 5) Aggregate by (weekday × week‑of‑month × hour) and pick best times

In [6]:

# Average predicted traffic for each (weekday, week_of_month, hour)
grp_cols = ["dow","week_of_month","hour"]
agg = (df_biz.groupby(grp_cols)["predicted"].mean().reset_index())

# Keep valid weeks 1..5
agg = agg[(agg["week_of_month"]>=1) & (agg["week_of_month"]<=5)].copy()
agg["Weekday"] = agg["dow"].map(dict(enumerate(weekday_names)))
agg["Week"] = agg["week_of_month"].map({1:"1st",2:"2nd",3:"3rd",4:"4th",5:"5th"})
agg["PrettyTime"] = agg["hour"].apply(to_ampm_from_hour)

# Sort by lowest predicted traffic
agg = agg.sort_values(["predicted","dow","week_of_month","hour"]).reset_index(drop=True)

# Top K overall times (lowest traffic) within business hours
TOP_K = int(TOP_K)
top_all = agg.nsmallest(TOP_K, "predicted").copy()
top_all["Expected Traffic"] = top_all["predicted"].round(1)
top_all_display = top_all[["Weekday","Week","PrettyTime","Expected Traffic"]].reset_index(drop=True)

# Tie logic for final recommendation(s)
best_val = float(top_all.iloc[0]["predicted"])
within = top_all[np.abs(top_all["predicted"] - best_val) <= float(TIE_EPS)].copy()
if len(within) == 1:
    final = within.iloc[[0]].copy()  # single best
    tie_note = None
else:
    final = within.iloc[:TOP_K].copy()  # up to 4 if tied
    tie_note = f"Multiple options within ±{TIE_EPS} traffic units of the best score."

def human_sentence(row):
    return f"Best time to visit: {row['Weekday']}, {row['Week']} week at {row['PrettyTime']} (expected traffic ≈ {row['predicted']:.1f})."

sentences = [human_sentence(r) for _, r in final.iterrows()]

# Pattern note: if all best share same weekday
weekday_counts = final["Weekday"].value_counts()
pattern_note = None
if len(weekday_counts) == 1 and len(final) > 1:
    pattern_note = f"Pattern insight: Best slots all fall on **{weekday_counts.index[0]}s**."

top_all_display


,Weekday,Week,PrettyTime,Expected Traffic
0,Wednesday,4th,3:00 PM,34.3
1,Thursday,5th,3:00 PM,34.4
2,Friday,5th,3:00 PM,35.4
3,Wednesday,1st,3:00 PM,36.2


## 6) Pretty Text Dashboard + Save to TXT

In [7]:

line = "—"*64
print("\n" + line)
print(" BEST TIME TO VISIT — DASHBOARD (Hourly; Business Hours)")
print(line)
print(f" Business hours: {BUSINESS_START} → {BUSINESS_END}  |  Granularity: Hourly")
print(f" Model test MAE: {mae:.2f}   R²: {r2:.3f}")
print(line)

if tie_note:
    print(" Final Recommendation: (near tie)")
else:
    print(" Final Recommendation:")
print("  •", sentences[0])
if len(sentences) > 1:
    print("\n Other near‑equal options:")
    for s in sentences[1:]:
        print("  •", s)
if pattern_note:
    print("\n " + pattern_note)
print(line)

# Save dashboard text
buf = io.StringIO()
_stdout = sys.stdout
sys.stdout = buf
print("\n" + line)
print(" BEST TIME TO VISIT — DASHBOARD (Hourly; Business Hours)")
print(line)
print(f" Business hours: {BUSINESS_START} → {BUSINESS_END}  |  Granularity: Hourly")
print(f" Model test MAE: {mae:.2f}   R²: {r2:.3f}")
print(line)
if tie_note:
    print(" Final Recommendation: (near tie)")
else:
    print(" Final Recommendation:")
print("  •", sentences[0])
if len(sentences) > 1:
    print("\n Other near‑equal options:")
    for s in sentences[1:]:
        print("  •", s)
if pattern_note:
    print("\n " + pattern_note)
print(line)
sys.stdout = _stdout
with open("best_time_dashboard.txt", "w", encoding="utf-8") as f:
    f.write(buf.getvalue())
print("Saved: best_time_dashboard.txt")



————————————————————————————————————————————————————————————————
 BEST TIME TO VISIT — DASHBOARD (Hourly; Business Hours)
————————————————————————————————————————————————————————————————
 Business hours: 09:00 → 17:00  |  Granularity: Hourly
 Model test MAE: 6.66   R²: 0.664
————————————————————————————————————————————————————————————————
 Final Recommendation: (near tie)
  • Best time to visit: Wednesday, 4th week at 3:00 PM (expected traffic ≈ 34.3).

 Other near‑equal options:
  • Best time to visit: Thursday, 5th week at 3:00 PM (expected traffic ≈ 34.4).
————————————————————————————————————————————————————————————————
Saved: best_time_dashboard.txt


## 7) Plotly Mini Dashboard (Line Chart + Top Options Table)

In [8]:

import plotly.graph_objs as go

# Line chart (Observed vs Predicted)
hist = df_biz.copy()
fig_line = go.Figure()
fig_line.add_trace(go.Scatter(
    x=hist["timestamp"], y=hist["traffic"], mode="lines", name="Observed"
))
fig_line.add_trace(go.Scatter(
    x=hist["timestamp"], y=hist["predicted"], mode="lines", name="Predicted", opacity=0.6
))
fig_line.update_layout(
    title="Observed vs Predicted Traffic (Business Hours)",
    xaxis_title="Time", yaxis_title="Traffic", template="plotly_white", height=420
)

# Table of final recommendations (1..4 rows)
table_df = final[["Weekday","Week","PrettyTime","predicted"]].copy()
table_df.rename(columns={"predicted":"Expected Traffic"}, inplace=True)
table_df["Expected Traffic"] = table_df["Expected Traffic"].round(1)

fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(table_df.columns), fill_color="#2a9d8f", font=dict(color="white", size=12), align="left"),
    cells=dict(values=[table_df[c] for c in table_df.columns], fill_color="lavender", align="left")
)])
fig_table.update_layout(title="Final Recommendation(s)", template="plotly_white", height=280)

fig_line.show()
fig_table.show()
